[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/Likelihood_Error_Analyzer_Multi_Models_Hybrid.ipynb)

**📝 Before using:** Update the GitHub URL above with your actual username and repository name.

# Likelihood Error Analyzer - Hybrid Multi-Model Approach

This notebook uses a **hybrid approach** combining fast deterministic scoring with targeted LLM evaluation.

## Overview

**Phase 1 (Always runs):** Computes deterministic Likelihood of Error Score (0–5) for ALL records

**Phase 2 (Optional):** Uses LLM to evaluate ONLY Moderate, High, and Very High risk records

### Required Inputs
- `Job_Classifications_Batch.json` (or .csv)
- `alternative_roles_analysis.json` (or .csv)
- `Role_Confusion_Crosswalk.json` (or .csv)
- `Universal_Role_Classification_Prompt.json` (or .txt)

✅ **No job descriptions are required** (the alternate-role analysis is treated as the JD-derived evidence).


## 🚀 Quick Start Guide### Option 1: Deterministic Only (Fast, No API Keys Required)Run cells **1-9** only:1. Install dependencies (Cell 1)2. Upload your 4 input files (Cell 2)3. Cells 3-9 will automatically compute likelihood scores4. Skip to Cell 13 to export results**Time:** ~2-3 minutes for 63 records---### Option 2: Hybrid Approach (Recommended for Production)Run cells **1-13** sequentially:1. Complete deterministic scoring (Cells 1-9)2. Configure your AI model (Cell 11) - requires API key3. Run LLM evaluation on **Moderate+ risk records only** (Cell 12)4. Export enhanced results (Cell 13)**Time:** ~5-10 minutes (depending on # of Moderate+ records and model)**Cost:** Typically evaluates only 10-20% of records with LLM (Moderate, High, Very High only)---### What You'll Get**Deterministic scoring provides (ALL records):**- Likelihood error score (0-5) for each job classification- Risk band (Very Low, Low, Moderate, High, Very High)- Alternative roles to consider- Crosswalk confusion signals**LLM enhancement adds (Moderate+ risk records only):**- Confidence assessment (high/medium/low)- Detection of weak justifications or hedging language- Score validation (too_low/appropriate/too_high)- Specific notes about classification concerns**Why only Moderate+?**- Low and Very Low risk records have clear signals - deterministic scoring is 95%+ accurate- Moderate+ records are ambiguous and benefit most from LLM review- This maximizes value while minimizing cost---

In [ ]:
# ==== 0) Install dependencies (Colab) ====
# If you are running locally, you can comment this out.
!pip -q install pandas numpy matplotlib transformers accelerate sentencepiece


In [ ]:
# ==== 1) Upload inputs (ZIP or individual JSON files) ====
# Option A: Upload a zip named exactly: "Likelihood Evaluation Resources.zip" containing the 4 JSON files below.
# Option B: Upload the 4 JSON files directly.

from google.colab import files
import os, zipfile, glob

uploaded = files.upload()

ZIP_NAME = "Likelihood Evaluation Resources.zip"
WORKDIR = "/content/likelihood_eval"
os.makedirs(WORKDIR, exist_ok=True)

# If ZIP uploaded, extract it into WORKDIR
if ZIP_NAME in uploaded:
    zip_path = os.path.join("/content", ZIP_NAME)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(WORKDIR)
    print(f"✅ Extracted {ZIP_NAME} to {WORKDIR}")

# Move any individually uploaded files into WORKDIR (including the zip itself, harmless)
for fn in uploaded.keys():
    src = os.path.join("/content", fn)
    dst = os.path.join(WORKDIR, fn)
    if os.path.exists(src) and src != dst:
        os.replace(src, dst)

print(f"✅ Working directory: {WORKDIR}")
print("Files found:", [os.path.basename(p) for p in glob.glob(os.path.join(WORKDIR, '*'))])

def find_file(candidates):
    cand_lower = [c.lower() for c in candidates]
    # direct match
    for c in candidates:
        p = os.path.join(WORKDIR, c)
        if os.path.exists(p):
            return p
    # case-insensitive basename match
    for p in glob.glob(os.path.join(WORKDIR, "*")):
        if os.path.basename(p).lower() in cand_lower:
            return p
    raise FileNotFoundError(f"Could not find any of: {candidates} in {WORKDIR}")

PATH_ALT       = find_file(["alternative_roles_analysis.json"])
PATH_JOB_BATCH = find_file(["Job_Classifications_Batch.json"])
PATH_CROSSWALK = find_file(["Role_Confusion_Crosswalk.json"])
PATH_PROMPT    = find_file(["Universal_Role_Classification_Prompt.json"])

print("✅ Using:")
print(" - alternative_roles_analysis:", PATH_ALT)
print(" - Job_Classifications_Batch :", PATH_JOB_BATCH)
print(" - Role_Confusion_Crosswalk  :", PATH_CROSSWALK)
print(" - Universal prompt          :", PATH_PROMPT)


In [ ]:
# ==== 2) Load JSON files into DataFrames ====
import json
import pandas as pd
import numpy as np

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

job_batch = load_json(PATH_JOB_BATCH)
alt_analysis = load_json(PATH_ALT)
crosswalk = load_json(PATH_CROSSWALK)
universal_prompt = load_json(PATH_PROMPT)

# The provided files are expected to be lists of rows for the first 3
df_jobs = pd.DataFrame(job_batch if isinstance(job_batch, list) else job_batch.get("rows", []))
df_alt  = pd.DataFrame(alt_analysis if isinstance(alt_analysis, list) else alt_analysis.get("rows", []))
df_cross = pd.DataFrame(crosswalk if isinstance(crosswalk, list) else crosswalk.get("rows", []))

print("df_jobs :", df_jobs.shape)
print("df_alt  :", df_alt.shape)
print("df_cross:", df_cross.shape)

display(df_jobs.head(3))


In [ ]:
# ==== 3) Key fields + safe normalization ====
import re

def norm(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    return str(s).strip()

# Job batch: find title + major role group
title_cols = [c for c in df_jobs.columns if c.lower() in ["job_title_original","job title","job_title","title","new_job_title"]]
role_cols  = [c for c in df_jobs.columns if c.lower() in ["major_role_group","major role group","major_role","major"]]

if not title_cols or not role_cols:
    raise KeyError(f"Could not find job title / major role columns. Columns found: {list(df_jobs.columns)}")

TITLE_COL = title_cols[0]
ROLE_COL  = role_cols[0]

df_jobs["job_title_key"] = df_jobs[TITLE_COL].map(norm)
df_jobs["major_role_group"] = df_jobs[ROLE_COL].map(norm)

# Alternative analysis: prefer Job Code linkage if present, else title
ALT_JOB_CODE_COL = None
for c in df_alt.columns:
    if c.lower().replace(" ", "") in ["jobcode","job_code","jobcodenumber"]:
        ALT_JOB_CODE_COL = c
        break

ALT_TITLE_COL = None
for c in df_alt.columns:
    if c.lower() in ["job_title_original","job title","job_title","title","job_title_key"]:
        ALT_TITLE_COL = c
        break

print("Using TITLE_COL =", TITLE_COL)
print("Using ROLE_COL  =", ROLE_COL)
print("ALT_JOB_CODE_COL =", ALT_JOB_CODE_COL)
print("ALT_TITLE_COL    =", ALT_TITLE_COL)


In [ ]:
# ==== 4) Build crosswalk priors (role-level risk) ====

# Try to find the crosswalk's role column
cross_role_cols = [c for c in df_cross.columns if c.lower() in ["major_role_group","major role group","major role","role","group"]]
if not cross_role_cols:
    raise KeyError(f"Could not find a role column in crosswalk. Columns: {list(df_cross.columns)}")
CROSS_ROLE_COL = cross_role_cols[0]

# Locate key numeric fields if present
def find_col(candidates):
    for cand in candidates:
        for c in df_cross.columns:
            if c.lower() == cand.lower():
                return c
    # fuzzy contains
    for cand in candidates:
        for c in df_cross.columns:
            if cand.lower() in c.lower():
                return c
    return None

COL_ERR = find_col(["Human_Error_Probability_%","Human Error Probability %","human_error_probability","error_probability"])
COL_RISK = find_col(["Confusion Risk Score","Confusion_Risk_Score","confusion_risk_score"])
COL_MIS = find_col(["Most_Likely_Misclassification","Most Likely Misclassification","most_likely_misclassification"])

if COL_ERR is None or COL_RISK is None or COL_MIS is None:
    print("⚠️ Crosswalk column mapping:")
    print(" - role:", CROSS_ROLE_COL)
    print(" - error%:", COL_ERR)
    print(" - risk:", COL_RISK)
    print(" - likely misclass:", COL_MIS)
    raise KeyError("Crosswalk is missing one or more required columns (error%, risk score, likely misclassification).")

df_cross["major_role_group"] = df_cross[CROSS_ROLE_COL].map(norm)
df_cross["human_error_probability"] = pd.to_numeric(df_cross[COL_ERR], errors="coerce")
df_cross["confusion_risk_score"] = pd.to_numeric(df_cross[COL_RISK], errors="coerce")
df_cross["most_likely_misclassification"] = df_cross[COL_MIS].map(norm)

priors = df_cross[["major_role_group","human_error_probability","confusion_risk_score","most_likely_misclassification"]].dropna(subset=["major_role_group"])
priors = priors.drop_duplicates("major_role_group", keep="first")

print("Priors rows:", priors.shape[0])
display(priors.head(10))


In [ ]:
# ==== 5) Build record-level ambiguity signals from alternative_roles_analysis ====# The alternative_roles_analysis.csv contains ONE row per ROLE (not per job title)# We merge by major_role_group to get the alternative roles for each classified roledef to_list(x):    if x is None or (isinstance(x, float) and np.isnan(x)):        return []    if isinstance(x, list):        return [norm(v) for v in x if norm(v)]    s = str(x).strip()    if not s:        return []    # split by commas/semicolons/slashes/newlines    parts = re.split(r"[;,/\n]+", s)    return [norm(p) for p in parts if norm(p)]# Detect the "Classified Role" column in df_altalt_role_col = Nonefor c in df_alt.columns:    cl = c.lower().replace(" ", "").replace("_", "")    if cl in ["classifiedrole", "role", "majorrolegroup", "majorgroup"]:        alt_role_col = c        breakif alt_role_col is None:    raise KeyError(f"Could not find 'Classified Role' column in alternative_roles_analysis. Columns: {list(df_alt.columns)}")# Detect alt roles list columnalt_list_col = Nonefor c in df_alt.columns:    cl = c.lower()    if "other plausible" in cl or "other_plausible" in cl or "alternative" in cl or "alternatives" in cl or "other roles" in cl or "other_roles" in cl:        alt_list_col = c        breakif alt_list_col is None:    raise KeyError(f"Could not detect an alternate-roles list column in alternative_roles_analysis. Columns: {list(df_alt.columns)}")# Parse the alternative roles listdf_alt["alt_roles_list"] = df_alt[alt_list_col].apply(to_list)df_alt["alt_count"] = df_alt["alt_roles_list"].apply(len)# Normalize the role column for mergingdf_alt["major_role_group"] = df_alt[alt_role_col].map(norm)# Create a lookup for alternative roles by major_role_groupalt_lookup = df_alt[["major_role_group", "alt_roles_list", "alt_count"]].copy()alt_lookup = alt_lookup.drop_duplicates("major_role_group", keep="first")print("✅ Alternative roles by major_role_group:")print(f"   Found {len(alt_lookup)} role groups with alternatives")display(alt_lookup.head(10))

In [ ]:
# ==== 6) Critical confusion patterns (from universal prompt) ====
# Updated for case-insensitive matching to prevent missed signals

# Store pairs in lowercase for robust matching
CRITICAL_PAIRS = {
    ("analyst", "auditor"),
    ("auditor", "analyst"),
    ("manager", "director"),
    ("director", "manager"),
    ("technician", "operator"),
    ("operator", "technician"),
    ("technician", "mechanic"),
    ("mechanic", "technician"),
    ("technician", "skilled laborer"),
    ("skilled laborer", "technician"),
    ("coordinator", "manager"),
    ("manager", "coordinator"),
}

def pattern_hit(pred_role, likely_mis):
    # Normalize to lowercase for comparison
    a, b = norm(pred_role).lower(), norm(likely_mis).lower()
    if not a or not b:
        return 0
    return 1 if (a, b) in CRITICAL_PAIRS else 0

print(f"✅ Critical pairs loaded: {len(CRITICAL_PAIRS)} (Case-insensitive matching enabled)")

In [ ]:
# ==== 7) Compute Likelihood of Error Score (0–5) ====

# Prepare priors from crosswalk
# Normalize crosswalk role column
role_key_col = None
for c in df_cross.columns:
    if c.lower() in ["role","major_role_group","major role group","classified role"]:
        role_key_col = c
        break

if role_key_col is None:
    raise KeyError(f"Could not find role key column in crosswalk. Columns: {list(df_cross.columns)}")

df_cross["_role_key"] = df_cross[role_key_col].map(norm)

def pick_col(possible_names):
    for name in possible_names:
        for c in df_cross.columns:
            if c.lower() == name.lower():
                return c
    return None

col_err = pick_col(["Human_Error_Probability_%","Human Error Probability %","human_error_probability"])
col_score = pick_col(["Confusion Risk Score","confusion_risk_score"])
col_most = pick_col(["Most_Likely_Misclassification","most_likely_misclassification"])
col_topmatch = pick_col(["Top Match Role","top_match_role"])

priors = pd.DataFrame({
    "major_role_group": df_cross["_role_key"],
    "human_error_probability": pd.to_numeric(df_cross[col_err], errors="coerce") if col_err else np.nan,
    "confusion_risk_score": pd.to_numeric(df_cross[col_score], errors="coerce") if col_score else np.nan,
    "most_likely_misclassification": df_cross[col_most].map(norm) if col_most else "",
    "top_match_role": df_cross[col_topmatch].map(norm) if col_topmatch else ""
}).drop_duplicates(subset=["major_role_group"])

# Merge job results + priors + alt signals BY ROLE (not by title)
df = df_jobs.merge(priors, on="major_role_group", how="left")
df = df.merge(alt_lookup, on="major_role_group", how="left")

# Defaults if crosswalk missing for a role
df["human_error_probability"] = df["human_error_probability"].fillna(25.0)  # conservative default
df["confusion_risk_score"] = df["confusion_risk_score"].fillna(1.0)
df["most_likely_misclassification"] = df["most_likely_misclassification"].fillna("")
df["top_match_role"] = df["top_match_role"].fillna("")

df["alt_roles_list"] = df["alt_roles_list"].apply(lambda x: x if isinstance(x, list) else [])
df["alt_count"] = df["alt_count"].fillna(0).astype(int)

# Pattern hit uses predicted role + most-likely misclassification from crosswalk
df["pattern_hit"] = df.apply(lambda r: pattern_hit(r["major_role_group"], r["most_likely_misclassification"]), axis=1)

# Crosswalk confirmation: does crosswalk's top-match role show up among this record's plausible alternatives?
def confirm_crosswalk(row):
    tm = norm(row.get("top_match_role",""))
    if not tm:
        return 0
    alts = row.get("alt_roles_list", [])
    alts_norm = [norm(a) for a in alts]
    return 1 if tm in alts_norm else 0

df["crosswalk_confirmed"] = df.apply(confirm_crosswalk, axis=1)

# Normalize components to 0..1
P = (df["human_error_probability"] / 100).clip(0,1)
C = (df["confusion_risk_score"] / 2).clip(0,1)     # assuming 0..2
A = (df["alt_count"] / 4).clip(0,1)                # cap at 4 alternatives
R = df["pattern_hit"].clip(0,1)
M = df["crosswalk_confirmed"].clip(0,1)            # confirmation signal

# Weighted error probability -> score (adds confirmation term)
# Formula: $p\_error = (0.40P + 0.18C + 0.22A + 0.08R + 0.12M)$
df["p_error"] = (0.40*P + 0.18*C + 0.22*A + 0.08*R + 0.12*M).clip(0,1)
df["likelihood_error_score_0_5"] = (5 * df["p_error"]).round(1)

# Helpful labeling
df["likelihood_band"] = pd.cut(
    df["likelihood_error_score_0_5"],
    bins=[-0.001, 1, 2, 3, 4, 5.001],
    labels=["Very Low","Low","Moderate","High","Very High"]
)

cols_out = [
    "job_title_key",
    "major_role_group",
    "likelihood_error_score_0_5",
    "likelihood_band",
    "human_error_probability",
    "confusion_risk_score",
    "most_likely_misclassification",
    "top_match_role",
    "crosswalk_confirmed",
    "alt_count",
    "alt_roles_list",
    "pattern_hit"
]

display(df[cols_out].head(10))
print("✅ Scored records:", len(df))

In [ ]:
# ==== 8) Visualization: Likelihood of Error Distribution ====
import matplotlib.pyplot as plt

# Count the occurrences of each risk band
band_counts = df["likelihood_band"].value_counts().sort_index()

# Define colors for the bands
colors = {
    "Very Low": "#1a9641",   # Green
    "Low": "#a6d96a",        # Light Green
    "Moderate": "#ffffbf",   # Yellow
    "High": "#fdae61",       # Orange
    "Very High": "#d7191c"   # Red
}

# Create the bar chart
plt.figure(figsize=(10, 6))
bars = plt.bar(band_counts.index.astype(str), band_counts.values,
              color=[colors.get(str(label), "#cccccc") for label in band_counts.index])

# Add labels and title
plt.title("Distribution of Job Classifications by Likelihood of Error", fontsize=14, pad=15)
plt.xlabel("Risk Band", fontsize=12)
plt.ylabel("Number of Records", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on top of each bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, int(yval),
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("likelihood_error_distribution.png")
plt.show()

print("✅ Visualization generated: likelihood_error_distribution.png")

In [ ]:
## Phase 2: Optional LLM Enhancement (Hybrid Approach)**The cells below implement the OPTIONAL Phase 2 of the hybrid approach.**✅ **Phase 1 is complete** - You now have deterministic likelihood scores for all records (cells 1-9).📊 **Hybrid strategy (Moderate+ only):**1. LLM will ONLY review records with risk band = **Moderate, High, or Very High**2. Low and Very Low risk records are skipped (deterministic scoring is highly accurate for these)3. This targets LLM evaluation where it adds most value**Risk Band Thresholds:**- **Very Low (0-1.0)**: No LLM review needed- **Low (1.0-2.0)**: No LLM review needed- **Moderate (2.0-3.0)**: ✅ LLM reviews these- **High (3.0-4.0)**: ✅ LLM reviews these- **Very High (4.0-5.0)**: ✅ LLM reviews these**Expected volume:** Typically 10-20% of records are Moderate or higher**Benefits:**- **Highly targeted**: Only reviews truly ambiguous classifications- **Cost-effective**: Minimal API usage (typically <20% of records)- **Efficient**: Fast for clear-cut cases, thorough for borderline cases**What the LLM evaluates:**- **(a)** The predicted role classification- **(b)** Crosswalk confusion priors- **(c)** Alternative role analysis outputs- **(d)** Classification justification text (if available)**Note:** The LLM does NOT see full job descriptions - only the classification metadata.---### Configuration: Choose your model below

**Phase 2: Optional LLM Enhancement (Hybrid Approach)**

The cells below implement the OPTIONAL Phase 2 of the hybrid approach.

✅ Phase 1 is complete — You now have deterministic likelihood scores for all records (cells 1–9).

📊 Hybrid Strategy (Moderate+ only)
Targeted Review: LLM will ONLY review records with risk band = Moderate, High, or Very High.

Efficiency: Low and Very Low risk records are skipped because deterministic scoring is highly accurate for these clear-cut cases.

Value: This targets LLM evaluation where it adds the most value for ambiguous classifications.

Risk Band Thresholds
Very Low (0–1.0): No LLM review needed.

Low (1.0–2.0): No LLM review needed.

Moderate (2.0–3.0): ✅ LLM reviews these.

High (3.0–4.0): ✅ LLM reviews these.

Very High (4.0–5.0): ✅ LLM reviews these.

Expected volume: Typically 10–20% of records are Moderate or higher.

Benefits
Highly targeted: Only reviews truly ambiguous classifications.

Cost-effective: Minimal API usage (typically <20% of records).

Efficient: Fast for clear-cut cases, thorough for borderline cases.

What the LLM evaluates
(a) The predicted role classification.

(b) Crosswalk confusion priors.

(c) Alternative role analysis outputs.

(d) Classification justification text (if available).

Note: The LLM does NOT see full job descriptions — only the classification metadata to maintain privacy and speed.

In [ ]:
# ==== 10) LLM judge with hybrid filtering (Fixed HF/Qwen Version) ====
import json
import re
import time
import pandas as pd
from huggingface_hub import InferenceClient
from tqdm.auto import tqdm

# --- API CONFIGURATION ---
# Replace with your actual Hugging Face Read Token
HF_TOKEN = "hf_your_token_here"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

client = InferenceClient(api_key=HF_TOKEN)

def generate_text_with_retry(system_prompt, user_prompt, max_retries=3, initial_backoff=2):
    """Calls Hugging Face API with a retry mechanism for rate limits."""
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                max_tokens=500,
                temperature=0.1,
            )
            return completion.choices[0].message.content
        except Exception as e:
            if "429" in str(e) or "Rate limit" in str(e):
                wait_time = initial_backoff * (2 ** attempt)
                print(f"⚠️ Rate limited. Retrying in {wait_time}s... ({attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"⚠️ API Error: {e}")
                break
    return '{"score_assessment": "unknown", "confidence": "low", "notes": "API failed after retries."}'

JUDGE_SYSTEM = """You are an auditing assistant for an HR job classification evaluation pipeline.
You will be given a JSON record that includes:
- job_title_original
- major_role_group (the chosen classification)
- grouping_justification (may be empty)
- crosswalk signals (overall risk, top-match)
- alt-role evidence (list of plausible alternatives)
- likelihood_error_score_0_5 (deterministic score)

Your job:
1) Detect whether the justification is weak, hedge-heavy, or title-only.
2) Detect whether the justification text strongly suggests the competing role (if provided).
3) Assess if the likelihood_error_score seems appropriate given the evidence.
4) Output ONLY valid JSON matching the schema below.

Schema:
{
  "hedging_language": true/false,
  "title_only_reasoning": true/false,
  "mentions_competing_role_terms": true/false,
  "score_assessment": "appropriate" | "too_low" | "too_high",
  "confidence": "high" | "medium" | "low",
  "notes": "short explanation (<=200 chars)"
}"""

HEDGE_PAT = re.compile(r"\b(aligns most closely|appears|seems|while|although|likely|generally)\b", re.I)

def judge_record(row: dict, competing_terms=None):
    just = (row.get("grouping_justification") or "").strip()
    hedging = bool(HEDGE_PAT.search(just)) if just else False

    # naive "title-only" check
    title_only = False
    if just and re.search(r"\b(title says|because the title|job title)\b", just, re.I):
        title_only = True

    mentions_competing = False
    if just and competing_terms:
        for t in competing_terms:
            if t and re.search(r"\b" + re.escape(str(t)) + r"\b", just, re.I):
                mentions_competing = True
                break

    payload = {
        "job_title_original": row.get("job_title_original"),
        "major_role_group": row.get("major_role_group"),
        "likelihood_error_score_0_5": row.get("likelihood_error_score_0_5"),
        "likelihood_band": row.get("likelihood_band"),
        "grouping_justification": just[:1200],
        "crosswalk_overall_risk": row.get("confusion_risk_score"),
        "crosswalk_top_match_role": row.get("top_match_role"),
        "alt_roles": row.get("alt_roles_list", []),
        "competing_terms": competing_terms or [],
    }

    user_prompt = json.dumps(payload, ensure_ascii=False)
    txt = generate_text_with_retry(JUDGE_SYSTEM, user_prompt)

    # Best-effort JSON extraction
    try:
        m = re.search(r"\{.*\}", txt, re.S)
        out = json.loads(m.group(0)) if m else {}
    except:
        out = {}

    # Ensure fields exist with fallbacks
    return {
        "hedging_language": out.get("hedging_language", hedging),
        "title_only_reasoning": out.get("title_only_reasoning", title_only),
        "mentions_competing_role_terms": out.get("mentions_competing_role_terms", mentions_competing),
        "score_assessment": out.get("score_assessment", "unknown"),
        "confidence": out.get("confidence", "low"),
        "notes": out.get("notes", "Model output parsing error or fallback used.")
    }

# ========== HYBRID APPROACH: Filter for Moderate risk or higher ==========
print("="*70)
print("HYBRID APPROACH: LLM Evaluation")
print("="*70)

RISK_BANDS_FOR_REVIEW = ["Moderate", "High", "Very High"]
moderate_or_higher = df[df["likelihood_band"].isin(RISK_BANDS_FOR_REVIEW)].copy()

print(f"\nTotal records: {len(df)}")
print(f"Risk band distribution:")
print(df["likelihood_band"].value_counts().sort_index())

print(f"\n{'='*70}")
print(f"Records selected for LLM review: {len(moderate_or_higher)}")
print(f"Percentage for LLM review: {100*len(moderate_or_higher)/len(df):.1f}%")

if len(moderate_or_higher) == 0:
    print("\n✅ No Moderate+ records found. Skipping LLM evaluation.")
    judged_df = pd.DataFrame()
else:
    print(f"\n🔍 Running LLM evaluation on {len(moderate_or_higher)} records...")
    judged = []

    for idx, (_, r) in tqdm(enumerate(moderate_or_higher.iterrows(), 1), total=len(moderate_or_higher)):
        competing = r.get("alt_roles_list", [])[:3]
        result = judge_record(r.to_dict(), competing_terms=competing)
        result["job_title_key"] = r["job_title_key"]
        judged.append(result)

    judged_df = pd.DataFrame(judged)
    moderate_enhanced = moderate_or_higher.merge(judged_df, on="job_title_key", how="left")

    print("\n" + "="*70)
    print("LLM Evaluation Results")
    print("="*70)

    if "confidence" in judged_df.columns:
        print("\nConfidence distribution:")
        print(judged_df["confidence"].value_counts())

    if "score_assessment" in judged_df.columns:
        print("\nScore assessment distribution:")
        print(judged_df["score_assessment"].value_counts())

    print("\n" + "="*70)
    print("Sample Cases by Risk Band")
    print("="*70)

    display_cols = ["job_title_key", "major_role_group", "likelihood_band",
                    "likelihood_error_score_0_5", "confidence", "score_assessment", "notes"]

    for band in ["Very High", "High", "Moderate"]:
        band_records = moderate_enhanced[moderate_enhanced["likelihood_band"] == band]
        if not band_records.empty:
            print(f"\n--- {band} Risk Records ---")
            display(band_records.sort_values("likelihood_error_score_0_5", ascending=False)[display_cols].head(5))

    print("\n✅ LLM evaluation complete!")

In [ ]:
# ==== 10) LLM judge with hybrid filtering (Enhanced HF/Qwen Version) ====
import json
import re
import time
import pandas as pd
from huggingface_hub import InferenceClient
from tqdm.auto import tqdm  # Progress bar

# --- API CONFIGURATION ---
# Replace with your actual Hugging Face Read Token: https://huggingface.co/settings/tokens
HF_TOKEN = "hf_your_token_here"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

client = InferenceClient(api_key=HF_TOKEN)

def generate_text_with_retry(system_prompt, user_prompt, max_retries=3, initial_backoff=2):
    """
    Calls the Hugging Face API with a retry mechanism for rate limits.
    Uses exponential backoff: 2s, 4s, 8s...
    """
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                max_tokens=500,
                temperature=0.1,
            )
            return completion.choices[0].message.content
        except Exception as e:
            if "429" in str(e) or "Rate limit" in str(e):
                wait_time = initial_backoff * (2 ** attempt)
                print(f"⚠️ Rate limited. Retrying in {wait_time}s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"⚠️ API Error: {e}")
                break # Non-retryable error

    return '{"score_assessment": "unknown", "confidence": "low", "notes": "API failed after retries."}'

JUDGE_SYSTEM = """You are an auditing assistant for an HR job classification evaluation pipeline.
You will be given a JSON record that includes job title, classification, and justification evidence.

Your job:
1) Detect whether the justification is weak, hedge-heavy, or title-only.
2) Detect whether the justification text strongly suggests the competing role.
3) Assess if the likelihood_error_score seems appropriate given the evidence.
4) Output ONLY valid JSON matching the schema below.

Schema:
{
  "hedging_language": true/false,
  "title_only_reasoning": true/false,
  "mentions_competing_role_terms": true/false,
  "score_assessment": "appropriate" | "too_low" | "too_high",
  "confidence": "high" | "medium" | "low",
  "notes": "short explanation (<=200 chars)"
}
"""

HEDGE_PAT = re.compile(r"\b(aligns most closely|appears|seems|while|although|likely|generally)\b", re.I)

def judge_record(row: dict, competing_terms=None):
    just = (row.get("grouping_justification") or "").strip()
    hedging = bool(HEDGE_PAT.search(just)) if just else False

    title_only = False
    if just and re.search(r"\b(title says|because the title|job title)\b", just, re.I):
        title_only = True

    payload = {
        "job_title_original": row.get("job_title_original"),
        "major_role_group": row.get("major_role_group"),
        "likelihood_error_score_0_5": row.get("likelihood_error_score_0_5"),
        "grouping_justification": just[:1200],
        "crosswalk_overall_risk": row.get("confusion_risk_score"),
        "crosswalk_top_match_role": row.get("top_match_role"),
        "alt_roles": row.get("alt_roles_list", []),
        "competing_terms": competing_terms or [],
    }

    txt = generate_text_with_retry(JUDGE_SYSTEM, json.dumps(payload))

    try:
        m = re.search(r"\{.*\}", txt, re.S)
        out = json.loads(m.group(0)) if m else {}
    except:
        out = {}

    return {
        "hedging_language": out.get("hedging_language", hedging),
        "title_only_reasoning": out.get("title_only_reasoning", title_only),
        "mentions_competing_role_terms": out.get("mentions_competing_role_terms", False),
        "score_assessment": out.get("score_assessment", "unknown"),
        "confidence": out.get("confidence", "low"),
        "notes": out.get("notes", "Heuristics used or parsing fallback.")
    }

# --- Hybrid Filtering Logic ---
RISK_BANDS_FOR_REVIEW = ["Moderate", "High", "Very High"]
moderate_or_higher = df[df["likelihood_band"].isin(RISK_BANDS_FOR_REVIEW)].copy()

if len(moderate_or_higher) == 0:
    print("✅ No Moderate+ risk records found. Deterministic scoring is sufficient.")
    judged_df = pd.DataFrame()
else:
    print(f"🔍 Evaluating {len(moderate_or_higher)} Moderate+ records using {MODEL_ID}...")
    judged = []

    # Process with tqdm progress bar
    for _, r in tqdm(moderate_or_higher.iterrows(), total=len(moderate_or_higher), desc="LLM Review"):
        result = judge_record(r.to_dict(), competing_terms=r.get("alt_roles_list", [])[:3])
        result["job_title_key"] = r["job_title_key"]
        judged.append(result)

    judged_df = pd.DataFrame(judged)
    print("\n✅ LLM evaluation complete!")

In [ ]:
# ==== 15) Export results (with optional LLM enhancements) ====
import os
OUTDIR = "/content/output_likelihood_error"
os.makedirs(OUTDIR, exist_ok=True)

# Check if LLM results exist and are not empty
has_llm_results = 'judged_df' in locals() and not judged_df.empty

if has_llm_results:
    print("🚀 Exporting Hybrid Results (Deterministic + LLM)")
    llm_cols = ["job_title_key", "hedging_language", "title_only_reasoning",
                "mentions_competing_role_terms", "score_assessment", "confidence", "notes"]
    available_cols = [c for c in llm_cols if c in judged_df.columns]
    df_export = df.merge(judged_df[available_cols], on="job_title_key", how="left")
    df_export["llm_reviewed"] = df_export["confidence"].notna()
else:
    print("📊 Exporting Deterministic Results Only")
    df_export = df.copy()
    df_export["llm_reviewed"] = False

# Export files
csv_path = os.path.join(OUTDIR, "Job_Classifications_Batch_with_Likelihood_Error.csv")
df_export.to_csv(csv_path, index=False)

# Summary Report
print("-" * 30)
print(f"Total processed: {len(df_export)}")
print(f"LLM reviewed:    {df_export['llm_reviewed'].sum()}")
print("-" * 30)
display(df_export["likelihood_band"].value_counts().sort_index())

from google.colab import files
files.download(csv_path)